In [ ]:
import cv2
import gymnasium as gym
import ale_py
from gymnasium.wrappers import AtariPreprocessing, FrameStackObservation, RecordVideo

import os

import numpy as np

import random

from collections import deque

import torch
import torch.nn as nn
from torch.utils.tensorboard import SummaryWriter

import time
import matplotlib.pyplot as plt

In [34]:
train_video_dir = f"../data/dqn/training"

In [35]:
class ResizeRender(gym.Wrapper):
    def render(self):
        frame = self.env.render()

        return cv2.resize(
            frame,
            (640, 840),
            interpolation=cv2.INTER_NEAREST
        )

In [ ]:
train_episodes = 2000
training_period = train_episodes // 5
frameskips = 4

gym.register_envs(ale_py)
env = gym.make("ALE/Pacman-v5", frameskip=1, render_mode="rgb_array")
env = AtariPreprocessing(env, scale_obs=True, frame_skip=frameskips, terminal_on_life_loss=True)
env = ResizeRender(env)
env = FrameStackObservation(env, stack_size=frameskips)
env = RecordVideo(env, video_folder=train_video_dir, name_prefix="training",
    episode_trigger=lambda x: x % training_period == 0)

In [37]:
action_size = env.action_space.n

In [38]:
device = "cuda" if torch.cuda.is_available() else "cpu"
# device = "cpu"
device

'cuda'

In [39]:
def layer_init(layer, std=np.sqrt(2), bias_const=0.0):
    torch.nn.init.orthogonal_(layer.weight, std)
    torch.nn.init.constant_(layer.bias, bias_const)
    return layer

class PacmanAgent(nn.Module):
    def __init__(self, n_hid, n_out):
        super().__init__()
        self.network = nn.Sequential(
            layer_init(nn.Conv2d(frameskips, 32, kernel_size=8, stride=4)),
            nn.ReLU(),
            
            layer_init(nn.Conv2d(32, 64, kernel_size=4, stride=2)),
            nn.ReLU(),         
               
            layer_init(nn.Conv2d(64, 64, kernel_size=3, stride=1)),
            nn.ReLU(),
            
            nn.Flatten(),
            layer_init(nn.Linear(64 * 7 * 7, n_hid)),
            nn.ReLU(),  
        )
        
        self.actor = layer_init(nn.Linear(n_hid, n_out), std=0.01)
        self.critic = layer_init(nn.Linear(n_hid, 1), std=1)                

    def get_value(self, x):
        return self.critic(self.network(x))

    def get_action_and_value(self, x, action=None):
        hidden = self.network(x)
        logits = self.actor(hidden)
        probs = torch.distributions.Categorical(logits=logits)
        if action is None:
            action = probs.sample()
        return action, probs.log_prob(action), probs.entropy(), self.critic(hidden)
    
def train(
    env,
    model,
    optimizer,
    episodes,
    update_epochs,
    num_steps,
    target_kl = 0.03,
    mini_batch_size = 64,
    gae_lambda = 0.95,
    clip_coef = 0.2,
    ent_coef = 0.01,
    vf_coef = 0.5,
    max_grad_norm = 0.5,
    norm_advantages = False,
    clip_vloss = False,
    lr_decay = True,
    gamma = 0.99,
    initial_lr = 1e-3,
    seed=None):
    
    writer = SummaryWriter(f"runs/pacman")
    
    batch_size = num_steps
    
    obs = torch.zeros((num_steps, 1) + env.observation_space.shape).to(device)
    actions = torch.zeros((num_steps, 1) + env.action_space.shape, dtype=torch.long).to(device)
    logprobs = torch.zeros((num_steps, 1)).to(device)
    rewards = torch.zeros((num_steps, 1)).to(device)
    dones = torch.zeros((num_steps, 1)).to(device)
    values = torch.zeros((num_steps, 1)).to(device)
    
    global_step = 0
    start_time = time.time()
    
    for episode in range(1, episodes + 1):
        next_obs, info = env.reset(seed=seed)
        next_obs = torch.FloatTensor(next_obs).to(device)
        next_done = torch.zeros(1).to(device)
        
        total_reward = 0
        
        if lr_decay:
            frac = 1.0 - (episode - 1.0) / episodes
            lrnow = initial_lr * frac
            optimizer.param_groups[0]["lr"] = lrnow
        
        for step in range(0, num_steps):
            global_step += 1
            obs[step] = next_obs
            dones[step] = next_done
            
            with torch.no_grad():
                action, logprob, _, value = model.get_action_and_value(next_obs.unsqueeze(0))
                values[step] = value.flatten()
            actions[step] = action
            logprobs[step] = logprob
            
            next_obs, reward, terminations, truncations, infos = env.step(action.cpu().numpy().item())
            next_done = np.logical_or(terminations, truncations)
            rewards[step] = torch.tensor(reward).to(device).view(-1)
            total_reward += reward
            next_obs, next_done = torch.Tensor(next_obs).to(device), torch.tensor(next_done, dtype=torch.float32).to(device)
                        
        with torch.no_grad():
            next_value = model.get_value(next_obs.unsqueeze(0)).reshape(1, -1)
            advantages = torch.zeros_like(rewards).to(device)
            lastgaelam = 0
            for t in reversed(range(num_steps)):
                if t == num_steps - 1:
                    next_nonterminal = 1.0 - next_done
                    next_values = next_value
                else:
                    next_nonterminal = 1.0 - dones[t + 1]
                    next_values = values[t + 1]
                delta = rewards[t] + gamma * next_values * next_nonterminal - values[t]
                advantages[t] = lastgaelam = delta + gamma * gae_lambda * next_nonterminal * lastgaelam
            returns = advantages + values
            
        b_obs = obs.reshape((-1,) + env.observation_space.shape)
        b_logprobs = logprobs.reshape(-1)
        b_actions = actions.reshape((-1,) + env.action_space.shape)
        b_advanatages = advantages.reshape(-1)
        b_returns = returns.reshape(-1)
        b_values = values.reshape(-1)
        
        
        b_inds = np.arange(batch_size)
        clipfracs = []
        for epoch in range(update_epochs):
            np.random.shuffle(b_inds)
            for start in range(0, batch_size, mini_batch_size):
                end = start + mini_batch_size
                mb_inds = b_inds[start:end]
                
                _, newlogprob, entropy, newvalue = model.get_action_and_value(b_obs[mb_inds], b_actions.long()[mb_inds])
                logratio = newlogprob - b_logprobs[mb_inds]
                ratio = logratio.exp()
                
                with torch.no_grad():
                    old_approx_kl = (-logratio).mean()
                    approx_kl = ((ratio - 1) - logratio).mean()
                    clipfracs += [((ratio - 1.0).abs() > clip_coef).float().mean().item()]
                    
                mb_advantages = b_advanatages[mb_inds]
                if norm_advantages: 
                    mb_advantages = (mb_advantages-mb_advantages.mean()) / (mb_advantages.std() + 1e-8)
                
                pg_loss1 = -mb_advantages * ratio
                pg_loss2 = -mb_advantages * torch.clamp(ratio, 1 - clip_coef, 1 + clip_coef)
                pg_loss = torch.max(pg_loss1, pg_loss2).mean()
                
                new_value = newvalue.view(-1)

                if clip_vloss:
                    v_loss_unclipped = (
                        new_value - b_returns[mb_inds]
                    ) ** 2

                    v_clipped = (
                        b_values[mb_inds]
                        + torch.clamp(
                            new_value - b_values[mb_inds],
                            -clip_coef,
                            clip_coef,
                        )
                    )

                    v_loss_clipped = (
                        v_clipped - b_returns[mb_inds]
                    ) ** 2

                    v_loss_max = torch.max(
                        v_loss_unclipped,
                        v_loss_clipped
                    )

                    v_loss = 0.5 * v_loss_max.mean()

                else:
                    v_loss = 0.5 * (
                        (new_value - b_returns[mb_inds]) ** 2
                    ).mean()
                    
                entropy_loss = entropy.mean()
                loss = pg_loss - ent_coef * entropy_loss + v_loss * vf_coef
                
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
                optimizer.step()
                
            if target_kl is not None and approx_kl > target_kl:
                break
            
        y_pred, y_true = b_values.cpu().numpy(), b_returns.cpu().numpy()
        var_y = np.var(y_true)
        explained_var = np.nan if var_y == 0 else 1 - np.var(y_true - y_pred) / var_y
        
        writer.add_scalar("charts/learning_rate", optimizer.param_groups[0]["lr"], global_step)
        writer.add_scalar("losses/value_loss", v_loss.item(), global_step)
        writer.add_scalar("losses/policy_loss", pg_loss.item(), global_step)
        writer.add_scalar("losses/entropy", entropy_loss.item(), global_step)
        writer.add_scalar("losses/old_approx_kl", old_approx_kl.item(), global_step)
        writer.add_scalar("losses/approx_kl", approx_kl.item(), global_step)
        writer.add_scalar("losses/clipfrac", np.mean(clipfracs), global_step)
        writer.add_scalar("losses/explained_variance", explained_var, global_step)
        writer.add_scalar("charts/episode_reward",total_reward,global_step)
        print("SPS:", int(global_step / (time.time() - start_time)))
        writer.add_scalar("charts/SPS", int(global_step / (time.time() - start_time)), global_step)
            
    env.close()
    writer.close()

In [40]:
agent = PacmanAgent(512, action_size).to(device)
optimizer = torch.optim.NAdam(agent.parameters(), lr=1e-3)

train(
    env=env,
    model=agent,
    optimizer=optimizer,
    episodes=train_episodes,
    update_epochs=4,
    num_steps=512,
    target_kl=0.03,
    mini_batch_size=64,
    gae_lambda=0.95,
    clip_coef=0.2,
    ent_coef=0.01,
    vf_coef=0.5,
    max_grad_norm=0.5,
    norm_advantages=True,
    clip_vloss=True,
    lr_decay=True,
    gamma=0.99,
    initial_lr=1e-3,
    seed=42
)

SPS: 259
SPS: 283
SPS: 297
SPS: 311
SPS: 317
SPS: 325
SPS: 326
SPS: 331
SPS: 336
SPS: 333
SPS: 334
SPS: 334
SPS: 333
SPS: 332
SPS: 332
SPS: 333
SPS: 333
SPS: 334
SPS: 332
SPS: 332
SPS: 331
SPS: 331
SPS: 330
SPS: 329
SPS: 328
SPS: 327
SPS: 326
SPS: 327
SPS: 327
SPS: 329
SPS: 328
SPS: 328
SPS: 328
SPS: 329
SPS: 330
SPS: 330
SPS: 330
SPS: 329
SPS: 330
SPS: 329
SPS: 329
SPS: 329
SPS: 328
SPS: 328
SPS: 326
SPS: 326
SPS: 326
SPS: 327
SPS: 326
SPS: 327
SPS: 326
SPS: 326
SPS: 327
SPS: 327
SPS: 327
SPS: 327
SPS: 327
SPS: 327
SPS: 327
SPS: 327
SPS: 326
SPS: 326
SPS: 325
SPS: 325
SPS: 324
SPS: 324
SPS: 324
SPS: 323
SPS: 323
SPS: 323
SPS: 323
SPS: 322
SPS: 322
SPS: 321
SPS: 320
SPS: 320
SPS: 319
SPS: 319
SPS: 319
SPS: 319
SPS: 318
SPS: 317
SPS: 317
SPS: 316
SPS: 315
SPS: 314
SPS: 314
SPS: 313
SPS: 313
SPS: 313
SPS: 313
SPS: 312
SPS: 312
SPS: 312
SPS: 312
SPS: 311
SPS: 311
SPS: 311
SPS: 311
SPS: 310
SPS: 310
SPS: 310
SPS: 309
SPS: 309
SPS: 309
SPS: 309
SPS: 309
SPS: 309
SPS: 308
SPS: 308
SPS: 309
S

In [41]:
torch.save(agent.state_dict(), "pacman-agent.pth")
torch.save(optimizer.state_dict(), "pacman-optimizer.pth")